In [1]:
import time

import asyncio
import nest_asyncio

import pandas as pd

In [2]:
from src.heuristics.base_heuristics.non_fair.parallelise import mgreedy_optimised

from src import (
    optimised_welfare_grasp,
    water_filling_greedy_optimised,
)

from src.utils import Loader
from src.metrics import utility_gap
from src.diffusion_models import (
    estimate_cascade_by_community,
)

In [3]:
nest_asyncio.apply()

In [4]:
import networkx as nx
from pathlib import Path

path_to_networks = Path('src/data/synthetic/networks/')


async def load_multiple_graphs(sizes, prefix='barbasi_albert'):
    loader = Loader(max_workers=4)

    tasks = {
        size: loader.load(path_to_networks / f'{prefix}_{size}.pkl')
        for size in sizes
    }

    loaded_graphs = await asyncio.gather(*tasks.values())

    graphs = {}
    for size, graph in zip(tasks.keys(), loaded_graphs):
        costs = nx.get_node_attributes(graph, 'node_costs')
        graphs[size] = (graph, costs)

    return graphs

In [5]:
def run_modified_greedy(graph, costs, budget, probability, num_sims):
    return mgreedy_optimised(graph=graph, costs=costs, budget=budget, probability=probability, num_sims=num_sims)

def run_water_filling(graph, costs, budget, alpha, p, sims):
    return water_filling_greedy_optimised(graph=graph, costs=costs, budget=budget, alpha=alpha, probability=p, num_sims=sims)


def run_grasp(graph, costs, budget, alpha, p, sims):
    return optimised_welfare_grasp(
        graph=graph,
        costs=costs,
        budget=budget,
        alpha=0.5,
        welfare=alpha,
        propagation_rate=p,
        num_sims=sims
    )

In [6]:
def evaluate_baseline(algorithm_fn, graph, costs, budget, probability, num_sims):
    start_time = time.time()
    seeds = algorithm_fn(graph, costs, budget, probability=probability, num_sims=num_sims)
    runtime = time.time() - start_time

    frac = estimate_cascade_by_community(
        graph=graph,
        seeds=seeds,
        probability=probability,
        num_simulations=num_sims,
        random_state=42,
    )

    return {
        'spread': sum(frac.values()),
        'utility_gap': utility_gap(frac),
        'runtime_sec': runtime,
        'cost_used': sum(costs[n] for n in seeds)
    }

In [7]:
def evaluate_fair_algorithm(algorithm_fn, graph, costs, budget, alpha, probability, num_sims):
    start_time = time.time()
    seeds = algorithm_fn(graph, costs, budget, alpha, probability, num_sims)
    runtime = time.time() - start_time

    frac = estimate_cascade_by_community(
        graph=graph,
        seeds=seeds,
        probability=probability,
        num_simulations=num_sims,
        random_state=42,
    )

    return {
        'spread': sum(frac.values()),
        'utility_gap': utility_gap(frac),
        'runtime_sec': runtime,
        'cost_used': sum(costs[n] for n in seeds)
    }

In [8]:
def run_budget_sweep_and_save(
        graph,
        costs,
        size,
        output_dir,
        budget_fracs=None,
        alpha=0.0,
        p=0.25,
        sims=1000,
):
    if budget_fracs is None:
        budget_fracs = [0.005, 0.01, 0.05, 0.1, 0.3]

    total_cost = sum(costs.values())
    results = []

    baseline_spreads = {}
    for frac in budget_fracs:
        budget = frac * total_cost
        mg = evaluate_baseline(run_modified_greedy, graph, costs, budget, p, sims)
        baseline_spreads[frac] = {'mg': mg['spread']}

    algorithms = {
        'water_filling': run_water_filling,
        'grasp': run_grasp,
    }

    for frac in budget_fracs:
        budget = frac * total_cost
        base_mg_spread = baseline_spreads[frac]['mg']

        for name, func in algorithms.items():
            res = evaluate_fair_algorithm(func, graph, costs, budget, alpha, p, sims)
            res.update({
                'algorithm': name,
                'budget_frac': frac,
                'alpha': alpha,
                'pof_modified_greedy': 1 - res['spread'] / base_mg_spread,
            })
            results.append(res)

    df = pd.DataFrame(results)
    out_path = Path(output_dir) / f'size_{size}' / 'budget_sweep_results.csv'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    print(f'Saved budget sweep results for size {size} to {out_path}')

In [9]:
def run_alpha_sweep_and_save(
        graph,
        costs,
        size,
        output_dir,
        alpha_values=None,
        budget_frac=0.1,
        p=0.25,
        sims=1000,
):
    if alpha_values is None:
        alpha_values = [0.5, 0.1, 0.0, -3, -9]

    total_cost = sum(costs.values())
    budget = budget_frac * total_cost
    results = []

    # Evaluate baselines once (no fairness parameter used)
    base_mg = evaluate_baseline(run_modified_greedy, graph, costs, budget, p, sims)
    base_mg_spread = base_mg['spread']

    algorithms = {
        'water_filling': run_water_filling,
        'grasp': run_grasp,
    }

    for alpha in alpha_values:
        for name, func in algorithms.items():
            res = evaluate_fair_algorithm(func, graph, costs, budget, alpha, p, sims)
            res.update({
                'algorithm': name,
                'budget_frac': budget_frac,
                'alpha': alpha,
                'pof_modified_greedy': 1 - res['spread'] / base_mg_spread,
            })
            results.append(res)

    df = pd.DataFrame(results)
    out_path = Path(output_dir) / f'size_{size}' / 'alpha_sweep_results.csv'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    print(f'Saved alpha sweep results for size {size} to {out_path}')

In [10]:
graph_data = asyncio.run(load_multiple_graphs([1000, 5000, 10000]))

In [11]:
for size, (graph, costs) in graph_data.items():
    run_budget_sweep_and_save(
        graph=graph,
        costs=costs,
        size=size,
        output_dir='results/barbasi_albert/'
    )
    run_alpha_sweep_and_save(
        graph=graph,
        costs=costs,
        size=size,
        output_dir='results/barbasi_albert/'
    )

Initial Computation: 100%|██████████| 983/983 [00:03<00:00, 308.82it/s]
Seed Selection: 9it [00:24,  2.75s/it, seeds=9, influence=586.0210, budget_used=4.51/5.00, queue_size=889, inf_cache_hit=50.1%]
Initial Computation: 100%|██████████| 997/997 [00:03<00:00, 318.13it/s]
Seed Selection: 19it [00:28,  1.48s/it, seeds=19, influence=595.1630, budget_used=9.53/10.00, queue_size=974, inf_cache_hit=50.1%]
Initial Computation: 100%|██████████| 1000/1000 [00:03<00:00, 332.76it/s]
Seed Selection: 90it [00:30,  2.91it/s, seeds=90, influence=653.3170, budget_used=49.98/50.00, queue_size=905, inf_cache_hit=50.4%]
Initial Computation: 100%|██████████| 1000/1000 [00:03<00:00, 328.16it/s]
Seed Selection: 172it [00:31,  5.55it/s, seeds=172, influence=711.6490, budget_used=99.80/100.00, queue_size=828, inf_cache_hit=50.6%]
Initial Computation: 100%|██████████| 1000/1000 [00:03<00:00, 331.13it/s]
Seed Selection: 461it [00:33, 13.88it/s, seeds=461, influence=854.9060, budget_used=299.90/300.00, queue_siz

Setup complete: 1000 nodes, 18 communities


Initial Computation: 100%|██████████| 983/983 [00:03<00:00, 279.07it/s]
Seed Selection: 9it [00:27,  3.06s/it, seeds=9, welfare=-2813.6500, budget_used=4.68/5.00, queue_size=906, inf_cache_hit=0.0%, wel_cache_hit=50.1%]
Iterations: 100%|██████████| 50/50 [00:00<00:00, 345.82it/s, seeds=1, cost=4.3, welfare=-2814.186, inf_cache=0.0%, wel_cache=98.3%]


Setup complete: 1000 nodes, 18 communities


Initial Computation: 100%|██████████| 997/997 [00:03<00:00, 276.53it/s]
Seed Selection: 15it [00:29,  1.93s/it, seeds=15, welfare=-2813.5121, budget_used=9.53/10.00, queue_size=975, inf_cache_hit=0.0%, wel_cache_hit=50.1%]
Iterations: 100%|██████████| 50/50 [00:00<00:00, 312.75it/s, seeds=1, cost=8.5, welfare=-2813.910, inf_cache=0.0%, wel_cache=98.2%]


...

Setup complete: 10000 nodes, 14 communities


Initial Computation: 100%|██████████| 10000/10000 [14:46<00:00, 11.28it/s]
Seed Selection: 397it [34:48,  5.26s/it, seeds=397, welfare=-8934138287.4847, budget_used=999.70/1000.00, queue_size=9603, inf_cache_hit=0.0%, wel_cache_hit=50.3%]
Iterations: 100%|██████████| 50/50 [06:54<00:00,  8.29s/it, seeds=100, cost=963.4, welfare=-9482213563.828, inf_cache=0.0%, wel_cache=1.6%]


Setup complete: 10000 nodes, 14 communities


Initial Computation: 100%|██████████| 10000/10000 [14:28<00:00, 11.52it/s]
Seed Selection: 923it [39:07,  2.54s/it, seeds=923, welfare=-20950250134780305970628132864.0000, budget_used=999.77/1000.00, queue_size=9077, inf_cache_hit=0.0%, wel_cache_hit=50.6%]
Iterations: 100%|██████████| 50/50 [06:55<00:00,  8.30s/it, seeds=100, cost=972.8, welfare=-21959814349999362326701015040.000, inf_cache=0.0%, wel_cache=0.8%]


Saved alpha sweep results for size 10000 to results\barbasi_albert\size_10000\alpha_sweep_results.csv
